In [1]:
## Import the libraries

import os
import re
import pickle
import numpy as np
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

import fitz  
from pathlib import Path
from tqdm import tqdm


from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings




[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ahamm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ahamm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
d:\insurance-rag-chatbot\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## PDFs Loading and cleaning 

In [3]:
## LOADING PDFs

PDF_FOLDER = Path("D:\insurance-rag-chatbot\Data\insurance_documents")

pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))

print(f"Total PDFs: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

Total PDFs: 12
814 Insurance XII.pdf
DCOM309_INSURANCE_LAWS_AND_PRACTICES.pdf
English Health Handbook.pdf
Health Companion-Health Insurance Plan_GEN617.pdf
insurance-guidebook.pdf
Insurance_Handbook_20103.pdf
InsuranceLawandPractice.pdf
Life Insurance Handbook (English).pdf
Motor Insurance Handbook (English).pdf
SN_MPF_Eng.pdf
SN_P1_eng_2021.pdf
SN_P3_eng_2022.pdf


In [4]:
import os
from pathlib import Path
from collections import Counter

from dotenv import load_dotenv

from unstructured_client import UnstructuredClient
from unstructured_client.models import operations, shared

from unstructured.staging.base import elements_from_dicts
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document

# ------------------------------------------------------------------
# Load API Key
# ------------------------------------------------------------------
load_dotenv()

client = UnstructuredClient(
    api_key_auth=os.getenv("UNSTRUCTURED_API_KEY")
)

documents = []
total_raw_images = 0
total_chunk_images = 0

# ------------------------------------------------------------------
# Process PDFs
# ------------------------------------------------------------------
for pdf_path in pdf_files:

    pdf_name = Path(pdf_path).name
    print(f"Processing: {pdf_name}")

    with open(pdf_path, "rb") as f:

        request = operations.PartitionRequest(
            partition_parameters=shared.PartitionParameters(
                files=shared.Files(
                    content=f.read(),
                    file_name=pdf_name,
                ),
                strategy=shared.Strategy.HI_RES,
                coordinates=True,
                infer_table_structure=True,   # needed for text_as_html on tables
            )
        )

    response = client.general.partition(request=request)

    # --------------------------------------------------------------
    # Convert API response -> Unstructured Elements
    # --------------------------------------------------------------
    elements = elements_from_dicts(response.elements)

    # Count images BEFORE chunking, to check later if chunking drops them
    raw_image_count = len(
        [e for e in elements if getattr(e, "category", None) == "Image"]
    )
    total_raw_images += raw_image_count
    print(f"  Raw image elements (pre-chunk): {raw_image_count}")

    # --------------------------------------------------------------
    # Chunk using document titles
    # --------------------------------------------------------------
    chunks = chunk_by_title(
        elements,
        combine_text_under_n_chars=200,
        new_after_n_chars=800,
        max_characters=1000,
        multipage_sections=False,
    )

    # --------------------------------------------------------------
    # Convert chunks -> LangChain Documents
    # --------------------------------------------------------------
    for chunk in chunks:

        metadata = (
            chunk.metadata.to_dict()
            if hasattr(chunk.metadata, "to_dict")
            else {}
        )

        metadata["source"] = pdf_name
        metadata["category"] = chunk.category

        documents.append(
            Document(
                page_content=chunk.text.strip(),
                metadata=metadata,
            )
        )

# ------------------------------------------------------------------
# Statistics
# ------------------------------------------------------------------
print("\n==============================")
print(f"Total Documents : {len(documents)}")

lengths = [len(doc.page_content) for doc in documents]

print(f"Min Length      : {min(lengths)}")
print(f"Max Length      : {max(lengths)}")
print(f"Average Length  : {sum(lengths)/len(lengths):.0f}")

print("\nCategory Counts")
print("----------------")
print(Counter(doc.metadata.get("category") for doc in documents))

# ------------------------------------------------------------------
# Check Table metadata
# ------------------------------------------------------------------
table_docs = [
    doc
    for doc in documents
    if doc.metadata.get("category") == "Table"
]

print(f"\nTable Chunks : {len(table_docs)}")

if table_docs:
    print("\nFirst Table")
    print("-" * 80)
    print(table_docs[0].page_content[:500])

    if "text_as_html" in table_docs[0].metadata:
        print("\nHTML Available : YES")
        print(table_docs[0].metadata["text_as_html"][:500])
    else:
        print("\nHTML Available : NO")

# ------------------------------------------------------------------
# Check Image metadata -- and verify chunking didn't drop images
# ------------------------------------------------------------------
image_docs = [
    doc
    for doc in documents
    if doc.metadata.get("category") == "Image"
]

total_chunk_images = len(image_docs)

print(f"\nImage Chunks (post-chunk) : {total_chunk_images}")
print(f"Image Elements (pre-chunk, summed across files) : {total_raw_images}")

if total_chunk_images < total_raw_images:
    print(">>> WARNING: chunk_by_title appears to be dropping or absorbing "
          "some Image elements. Consider filtering Image elements out "
          "before chunking and re-adding them afterward.")
else:
    print(">>> Images preserved correctly through chunking.")

if image_docs:
    print("\nFirst Image")
    print("-" * 80)
    print(image_docs[0].page_content[:500])

# ------------------------------------------------------------------
# Preview
# ------------------------------------------------------------------
print("\n==============================")
print("Sample Documents")
print("==============================")

for doc in documents[:5]:

    print("-" * 80)
    print("Category :", doc.metadata.get("category"))
    print("Length   :", len(doc.page_content))
    print("Page     :", doc.metadata.get("page_number"))
    print(doc.page_content[:300])

Processing: 814 Insurance XII.pdf


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"


  Raw image elements (pre-chunk): 4
Processing: DCOM309_INSURANCE_LAWS_AND_PRACTICES.pdf


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "H

  Raw image elements (pre-chunk): 201
Processing: English Health Handbook.pdf


...

  Raw image elements (pre-chunk): 7
Processing: SN_P1_eng_2021.pdf


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "H

  Raw image elements (pre-chunk): 0
Processing: SN_P3_eng_2022.pdf


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "H

  Raw image elements (pre-chunk): 3

Total Documents : 6558
Min Length      : 1
Max Length      : 1000
Average Length  : 580

Category Counts
----------------
Counter({'CompositeElement': 6390, 'TableChunk': 162, 'Table': 6})

Table Chunks : 6

First Table
--------------------------------------------------------------------------------
Period Rate Sr. No. 1 For a period not exceeding 15 days 10% of Annual Rate 2 ------------------- do ---------------- 1 Month 15% of Annual Rate 3 ------------------- do ---------------- 2 Months 30% of Annual Rate 4 ------------------- do ---------------- 3 Months 40% of Annual Rate 5 ------------------- do ---------------- 4 Months 50% of Annual Rate 6 ------------------- do ---------------- 5 Months 60% of Annual Rate 7 ------------------- do ---------------- 6 Months 70% of Annual Rate 8 ---

HTML Available : YES
<table><tr><td>Sr. No.</td><td>Period</td><td/><td>Rate</td></tr><tr><td/><td>For a period not exceeding</td><td>15 days</td><td>10% of Ann

In [5]:
import pickle
import os

output_dir = r"D:\insurance-rag-chatbot\artifacts"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "documents.pkl")

with open(output_file, "wb") as f:
    pickle.dump(documents, f)

print(f"Saved {len(documents)} documents to:")
print(output_file)

Saved 6558 documents to:
D:\insurance-rag-chatbot\artifacts\documents.pkl


In [2]:
import pickle

with open(r"D:\insurance-rag-chatbot\artifacts\documents.pkl", "rb") as f:
    documents = pickle.load(f)

print(f"Loaded {len(documents)} documents")

Loaded 6558 documents


In [3]:
small_chunks = [doc for doc in documents if len(doc.page_content.strip()) < 50]

print(f"Chunks < 50 characters: {len(small_chunks)}")

Chunks < 50 characters: 112


In [11]:
documents = [
    doc
    for doc in documents
    if len(doc.page_content.strip()) >= 50
]

print(f"Remaining documents: {len(documents)}")

Remaining documents: 6446


## Embedding Model

In [7]:
# Extract text from LangChain Documents

text_chunks = [doc.page_content for doc in documents]


In [6]:
## Generating  embeddings :

model = SentenceTransformer("BAAI/bge-base-en-v1.5")

embeddings = model.encode(
    text_chunks,
    normalize_embeddings=True,
    batch_size=64, # Based on the hardware . Change prefered batch size  for better memeory optimization
    show_progress_bar=True
)

Batches:   1%|          | 1/101 [00:58<1:37:43, 58.63s/it]


KeyboardInterrupt: 

In [23]:
## saving the  embedding :

# Convert to NumPy float32 array
embeddings = np.asarray(embeddings, dtype=np.float32)

save_path = r"D:\insurance-rag-chatbot\artifacts\embeddings.npy"

# Save
np.save(save_path, embeddings)

print("Saved successfully!")
print("Shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)

Saved successfully!
Shape: (6446, 768)
Dtype: float32


In [24]:
## Load the embeddings :

save_path = r"D:\insurance-rag-chatbot\artifacts\embeddings.npy"

loaded_embeddings = np.load(save_path)

print("Loaded successfully!")
print("Shape:", loaded_embeddings.shape)
print("Dtype:", loaded_embeddings.dtype)


Loaded successfully!
Shape: (6446, 768)
Dtype: float32


## Pinecone Vector DB

In [ ]:
## connect the Qdrant DB:

import os
from dotenv import load_dotenv

load_dotenv()

## Qdrant API 

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")



if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found. Check your .env file")

## Groq API 

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Check your .env file")

In [ ]:
print("API key loaded :", PINECONE_API_KEY is not None)
print("API key length :", len(PINECONE_API_KEY))

API key loaded : True
API key length : 74


In [25]:
## VB client setup :

from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "insurance-rag"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"   # Change to your Pinecone region
        )
    )

print("Index is ready!")

Index is ready!


In [28]:
index = pc.Index("insurance-rag")

In [33]:
vectors = []

for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
    vectors.append({
        "id": f"doc-{i}",
        "values": embedding.tolist(),
        "metadata": {
            "text": doc.page_content,
            "category": doc.metadata.get("category"),
            "page_number": doc.metadata.get("page_number"),
            "filename": doc.metadata.get("filename"),
            "source": doc.metadata.get("source"),
        }
    })

In [34]:
batch_size = 100

for i in range(0, len(vectors), batch_size):
    batch = vectors[i:i + batch_size]
    index.upsert(vectors=batch)

print("Upload completed!")

INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into namespace ''
INFO: Upserting 100 vectors into n

Upload completed!


In [35]:
stats = index.describe_index_stats()
print(stats)

INFO: Describing index stats


DescribeIndexStatsResponse(dimension=768, total_vector_count=6446, metric='cosine', namespaces=1)


In [41]:
query = "What is motor vehicle insurance?"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
).tolist()

In [42]:
dense_results = index.query(
    vector=query_embedding,
    top_k=20,
    include_metadata=True
)

In [41]:
for i, match in enumerate(results["matches"], start=1):
    print("=" * 100)
    print(f"Rank      : {i}")
    print(f"Score     : {match['score']:.4f}")

    metadata = match["metadata"]

    print(f"Category  : {metadata.get('category')}")
    print(f"Page      : {metadata.get('page_number')}")
    print(f"File      : {metadata.get('filename')}")

    print("\nText:")
    print(metadata.get("text", "")[:500])

Rank      : 1
Score     : 0.8440
Category  : CompositeElement
Page      : 286
File      : InsuranceLawandPractice.pdf

Text:
Definition

“Health insurance is an insurance, which covers the financial loss arising out of poor health condition or due to permanent disability, which results in loss of income.”

A health insurance policy is a contract between an insurer and an individual or group, in which the insurer agrees to provide specified health insurance at an agreed upon price (premium). It usually provides either direct payment or reimbursement for expenses associated with illness and injuries. The cost and range 
Rank      : 2
Score     : 0.8334
Category  : CompositeElement
Page      : 5
File      : English Health Handbook.pdf

Text:
3 FAQs on Health Insurance

Q. Whatis Health Insurance?

3. FAQs on Health Insurance

Q. What is Health Insurance?

Ans. The term health insurance is a type of

insurance that covers your medical expenses.

available?

A health insurance policy is a c

## Sparse vector :

In [9]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "insurance-rag-sparse"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",          # Change to your region
        embed={
            "model": "pinecone-sparse-english-v0",
            "field_map": {
                "text": "chunk_text"
            }
        }
    )

print("Sparse index created successfully!")

sparse_index = pc.Index(index_name)

Sparse index created successfully!


In [10]:
stats = sparse_index.describe_index_stats()
print(stats)

DescribeIndexStatsResponse(total_vector_count=0, metric='dotproduct', namespaces=0)


In [12]:
records = []

for i, doc in enumerate(documents):

    records.append({
        "id": f"doc-{i}",
        "chunk_text": doc.page_content,
        "category": doc.metadata.get("category"),
        "page_number": doc.metadata.get("page_number"),
        "filename": doc.metadata.get("filename"),
        "source": doc.metadata.get("source")
    })

In [13]:
batch_size = 96

for i in range(0, len(records), batch_size):
    sparse_index.upsert_records(
        namespace="insurance",
        records=records[i:i + batch_size]
    )

print("Sparse documents uploaded successfully!")

Sparse documents uploaded successfully!


In [14]:
print(sparse_index.describe_index_stats())

DescribeIndexStatsResponse(total_vector_count=6446, metric='dotproduct', namespaces=1)


In [43]:
query = "What is motor vehicle insurance?"

sparse_results = sparse_index.search(
    namespace="insurance",
    query={
        "top_k": 20,
        "inputs": {
            "text": query
        }
    }
)

In [19]:
print(type(results))
print(results)

<class 'pinecone.models.vectors.search.SearchRecordsResponse'>
SearchRecordsResponse(result=SearchResult(hits=[Hit(id='doc-4128', score=12.070082664489746, fields={'category': 'CompositeElement', 'chunk_text': 'What Motor Insurance covers:\n\nThe damages to the vehicle due to the following perils are usually covered under OD section of the Motor Insurance policy:\n\n(a) Fire, Explosion, Self- Ignition, Lightning\n\n(b) Burglary/Housebreaking / Theft\n\n(c) Riot & Strike\n\n(d) Earthquake\n\n(e) Flood, Storm, Cyclone, Hurricane, tempest, inundation, hailstorm, frost\n\n(f) Accidental external means\n\n(g) Malicious Act\n\n(h) Terrorism acts\n\n(i) While in Transit by Rail/ Road, Inland waterways, Lift, Elevator or Air\n\n(j) Land slide / Rock slide', 'filename': 'InsuranceLawandPractice.pdf', 'page_number': 279.0, 'source': 'InsuranceLawandPractice.pdf'}), Hit(id='doc-4435', score=12.02841567993164, fields={'category': 'CompositeElement', 'chunk_text': 'What Motor Insurance covers\n\nTh

In [20]:
for i, hit in enumerate(results.result.hits, start=1):

    print("=" * 100)
    print(f"Rank      : {i}")
    print(f"Score     : {hit.score:.4f}")

    print(f"Category  : {hit.fields['category']}")
    print(f"Page      : {hit.fields['page_number']}")
    print(f"File      : {hit.fields['filename']}")

    print("\nText:")
    print(hit.fields["chunk_text"][:500])

Rank      : 1
Score     : 12.0701
Category  : CompositeElement
Page      : 279.0
File      : InsuranceLawandPractice.pdf

Text:
What Motor Insurance covers:

The damages to the vehicle due to the following perils are usually covered under OD section of the Motor Insurance policy:

(a) Fire, Explosion, Self- Ignition, Lightning

(b) Burglary/Housebreaking / Theft

(c) Riot & Strike

(d) Earthquake

(e) Flood, Storm, Cyclone, Hurricane, tempest, inundation, hailstorm, frost

(f) Accidental external means

(g) Malicious Act

(h) Terrorism acts

(i) While in Transit by Rail/ Road, Inland waterways, Lift, Elevator or Air

(j)
Rank      : 2
Score     : 12.0284
Category  : CompositeElement
Page      : 3.0
File      : Motor Insurance Handbook (English).pdf

Text:
What Motor Insurance covers

The damages to the vehicle due to the following perils are usually covered under OD section of the Motor Insurance policy

which would give a wider cover, including cover for

your vehicle.

a Fire, Explos

In [44]:
from collections import defaultdict

def reciprocal_rank_fusion(dense_hits, sparse_hits, k=60):

    scores = defaultdict(float)
    documents = {}

    # Dense
    for rank, doc in enumerate(dense_hits, start=1):
        doc_id = doc["id"]
        scores[doc_id] += 1 / (k + rank)
        documents[doc_id] = doc

    # Sparse
    for rank, doc in enumerate(sparse_hits, start=1):
        doc_id = doc["id"]
        scores[doc_id] += 1 / (k + rank)

        if doc_id not in documents:
            documents[doc_id] = doc

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    fused = []

    for doc_id, score in ranked:
        doc = documents[doc_id]
        doc["rrf_score"] = score
        fused.append(doc)

    return fused

In [45]:
dense_hits = []

for match in dense_results.matches:
    dense_hits.append({
        "id": match.id,
        "score": match.score,
        "metadata": match.metadata
    })

In [46]:
sparse_hits = []

for hit in sparse_results.result.hits:
    sparse_hits.append({
        "id": hit.id,
        "score": hit.score,
        "metadata": hit.fields
    })

In [47]:
rrf_results = reciprocal_rank_fusion(
    dense_hits,
    sparse_hits,
    k=60
)

In [48]:
for i, doc in enumerate(rrf_results[:10], start=1):

    print("=" * 100)
    print(f"Rank      : {i}")
    print(f"RRF Score : {doc['rrf_score']:.6f}")

    meta = doc["metadata"]

    print(f"Page      : {meta.get('page_number')}")
    print(f"File      : {meta.get('filename')}")
    print(f"Category  : {meta.get('category')}")

    text = meta.get("text") or meta.get("chunk_text")

    print("\nText:")
    print(text[:500])

Rank      : 1
RRF Score : 0.032266
Page      : 83
File      : 814 Insurance XII.pdf
Category  : CompositeElement

Text:
4.1. Introduction to Motor Vehicle Insurance

Motor Vehicle Insurance, also referred to as ‘Automotive Insurance’, is a contract of Insurance under which the Insurer indemnifies the Insured, who is the owner or an operator of a Motor Vehicle, against any loss that he may incur due to damage to the property (i.e. the Motor Vehicle) or any other person (i.e. Third Party) as a result of an accident.

There are two types of Motor Insurance viz:

A. Mandatory Motor Vehicle Insurance

B. Comprehensive
Rank      : 2
RRF Score : 0.031754
Page      : 277
File      : InsuranceLawandPractice.pdf
Category  : CompositeElement

Text:
DEFINITION

A motor insurance policy is a mandatory policy issued by an insurance company as part of prevention of public liability to protect the general public from any accident that might take place on the road. The law mandates that every owner of 

## Hybrid Serach Setup :


In [21]:
from qdrant_client.models import (
    VectorParams,
    SparseVectorParams,
    Distance,
)

In [162]:
## Config 

COLLECTION_NAME = "insurance_rag"

client.create_collection(
    collection_name=COLLECTION_NAME,

    vectors_config={
        "dense": VectorParams(
            size=768,
            distance=Distance.COSINE,
        )
    },

    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

print("Hybrid Collection Created Successfully!")
print(client.get_collections())

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `insurance_rag` already exists!"},"time":0.022491772}'

In [151]:
%pip install fastembed

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
## Create the Sparse vector for Hybrid search

from fastembed import SparseTextEmbedding

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/minicoil-v1"
)

sparse_embeddings = list(sparse_model.embed(text_chunks))

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]d:\insurance-rag-chatbot\venv\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ahamm\AppData\Local\Temp\fastembed_cache\models--Qdrant--minicoil-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 8 files: 100%|██████████| 8/8 [01:23<00:00, 

NameError: name 'text_chunks' is not defined

In [12]:
print(type(sparse_embeddings[0]))
print(sparse_embeddings[0])

NameError: name 'sparse_embeddings' is not defined

In [153]:
from qdrant_client.models import PointStruct, SparseVector
import uuid

points = []

for doc, dense_vector, sparse_vector in zip(
    chunk_documents,
    embeddings,
    sparse_embeddings,
):

    point = PointStruct(
        id=str(uuid.uuid4()),

        vector={
            "dense": dense_vector.tolist(),

            "sparse": SparseVector(
                indices=sparse_vector.indices.tolist(),
                values=sparse_vector.values.tolist(),
            ),
        },

        payload={
            "text": doc.page_content,
            "source": doc.metadata.get("source", ""),
            "page": doc.metadata.get("page", 0),
        },
    )

    points.append(point)

print(f"Total Points: {len(points)}")


Total Points: 5885


In [154]:
import time

BATCH_SIZE = 50

for i in range(0, len(points), BATCH_SIZE):

    while True:
        try:
            client.upsert(
                collection_name="insurance_rag",
                points=points[i:i+BATCH_SIZE],
                wait=True,
            )

            print(f"Uploaded {min(i+BATCH_SIZE, len(points))}/{len(points)}")
            break

        except Exception as e:
            print("Retrying...", e)
            time.sleep(5)

Uploaded 50/5885
Uploaded 100/5885
Uploaded 150/5885
Uploaded 200/5885
Uploaded 250/5885
Uploaded 300/5885
Uploaded 350/5885
Uploaded 400/5885
Uploaded 450/5885
Uploaded 500/5885
Uploaded 550/5885
Uploaded 600/5885
Uploaded 650/5885
Uploaded 700/5885
Uploaded 750/5885
Uploaded 800/5885
Uploaded 850/5885
Uploaded 900/5885
Uploaded 950/5885
Uploaded 1000/5885
Uploaded 1050/5885
Uploaded 1100/5885
Uploaded 1150/5885
Uploaded 1200/5885
Uploaded 1250/5885
Uploaded 1300/5885
Uploaded 1350/5885
Uploaded 1400/5885
Uploaded 1450/5885
Uploaded 1500/5885
Uploaded 1550/5885
Uploaded 1600/5885
Uploaded 1650/5885
Uploaded 1700/5885
Uploaded 1750/5885
Uploaded 1800/5885
Uploaded 1850/5885
Uploaded 1900/5885
Uploaded 1950/5885
Uploaded 2000/5885
Uploaded 2050/5885
Uploaded 2100/5885
Uploaded 2150/5885
Uploaded 2200/5885
Uploaded 2250/5885
Uploaded 2300/5885
Uploaded 2350/5885
Uploaded 2400/5885
Uploaded 2450/5885
Uploaded 2500/5885
Uploaded 2550/5885
Uploaded 2600/5885
Uploaded 2650/5885
Uploaded 2700

In [13]:
collection_info = client.get_collection("insurance_rag")

print("Points:", collection_info.points_count)

Points: 5885


In [14]:
print("Chunks :", len(chunk_documents))
print("Dense  :", len(embeddings))
print("Sparse :", len(sparse_embeddings))
print("Points :", collection_info.points_count)

NameError: name 'chunk_documents' is not defined

In [15]:
points, next_offset = client.scroll(
    collection_name="insurance_rag",
    limit=5,
    with_payload=True,
    with_vectors=False,
)

for point in points:
    print(point.id)
    print(point.payload["source"])
    print(point.payload["page"])
    print(point.payload["text"][:150])
    print("-" * 50)

001129b8-5f60-4397-aa48-98844ef6ab7f
SN_P3_eng_2022.pdf
102
5/23
In addition, an authorized insurer should provide a re-projection
of the policy loan amount upon request wi th a clear indication
that the intere
--------------------------------------------------
00169657-f8db-40a9-8270-55be7dba58e5
SN_MPF_Eng.pdf
37
3/1
3 KEY FEATURES OF THE MPF SYSTEM

The MPF System has a number of key features. This chapter explains the key features of
the MPF System.

3.1 SECU
--------------------------------------------------
00183d90-8a82-4a13-95f9-46d3c4e61119
DCOM309_INSURANCE_LAWS_AND_PRACTICES.pdf
61
Utmost Good Faith
Indemnity
Subrogation
Contribution
Insurable Interest
Sec. 10, Indian Contract Act,
No profit out of
insurance Insured
to be placed 
--------------------------------------------------
00185726-896e-4a5d-b697-a842f4e0eaf8
SN_P3_eng_2022.pdf
60
(vii) Compared with many existing indemnity hospital insurance
products, Certified Plans are more attractive in a number of ways,
as reflec

In [159]:
points, _ = client.scroll(
    collection_name="insurance_rag",
    limit=1,
    with_payload=True,
    with_vectors=True,
)

point = points[0]

print(point.vector.keys())

dict_keys(['dense', 'sparse'])


## Evaluation 

In [16]:
from importlib.metadata import version

print(version("qdrant-client"))

1.18.0


In [17]:
## query generated for dense and sparse :

query = "what is the amount claim for health insurance ?"

dense_query = model.encode(
    query,
    normalize_embeddings=True
).tolist()

sparse_query = list(sparse_model.embed([query]))[0]

NameError: name 'model' is not defined

In [229]:
from qdrant_client import models

results = client.query_points(
    collection_name="insurance_rag",
    prefetch=[
        models.Prefetch(
            query=dense_query,
            using="dense",
            limit=30,
        ),
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query.indices.tolist(),
                values=sparse_query.values.tolist(),
            ),
            using="sparse",
            limit= 10,
        ),
    ],
    query=models.FusionQuery(
        fusion=models.Fusion.RRF
    ),
    limit=10,
    with_payload=True,
)

In [230]:
print(type(results))
print(len(results.points))

<class 'qdrant_client.http.models.models.QueryResponse'>
10


In [231]:
print(results.points[0].payload)

{'text': '5.2. Coverage under Health Insurance\n\nA Health Insurance Policy would normally cover expenses reasona bly and necessarily\nincurred under the following heads in respect of each insured person subject to overall ceiling\nof sum insured (for all claims during one policy period).\n\nThus, all expenses incurred as part of treatment or hospitalization will be covered if:\n\n3⁄4 It is within the policy period\n3⁄4 Expenses covered are limited to the amount insured\n\nIn a health insurance policy the following may be covered:', 'source': '814 Insurance XII.pdf', 'page': 107}


In [210]:
%pip install -U transformers sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from sentence_transformers import CrossEncoder


reranker = CrossEncoder(
    "mixedbread-ai/mxbai-rerank-base-v2",
    device="cpu"   # or "cpu"
)

d:\insurance-rag-chatbot\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [226]:
pairs = [
    [query, point.payload["text"]]
    for point in results.points
]

scores = reranker.predict(pairs)

reranked_results = sorted(
    zip(scores, results.points),
    key=lambda x: x[0],
    reverse=True
)

In [227]:
for score, point in reranked_results:
    print(f"Cross Encoder Score: {score:.4f}")
    print(f"RRF Score: {point.score:.4f}")
    print(point.payload["text"][:200])
    print("-" * 80)

Cross Encoder Score: 7.8442
RRF Score: 0.2000
2
2. Health Insurance
The term ‘Health Insurance’ relates to a type of
insurance that essentially covers your medical
expenses. A health insurance policy like other policies is
a contract between an i
--------------------------------------------------------------------------------
Cross Encoder Score: 2.4342
RRF Score: 0.5000
5.2. Coverage under Health Insurance

A Health Insurance Policy would normally cover expenses reasona bly and necessarily
incurred under the following heads in respect of each insured person subject t
--------------------------------------------------------------------------------
Cross Encoder Score: 0.1431
RRF Score: 0.5000
lower. Claim free years can also be a factor in
determining the cost of the premium as it might
benefit you with certain percentage of discount.
This will automatically help you reduce your
premium.
Q
--------------------------------------------------------------------------------
Cross Encoder Sc

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant" # or  You can use  "llama-3.3-70b versatile" for better reasoning 
)



In [16]:
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.chat_history import InMemoryChatMessageHistory




In [17]:
from langchain_core.chat_history import InMemoryChatMessageHistory

SESSION_STORE = {}
MAX_MESSAGES = 6   

def get_session_history(session_id: str):
    if session_id not in SESSION_STORE:
        SESSION_STORE[session_id] = InMemoryChatMessageHistory()

    history = SESSION_STORE[session_id]

    # limit past history to control tokens
    
    if len(history.messages) > MAX_MESSAGES:
        history.messages = history.messages[-MAX_MESSAGES:]

    return history


In [18]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an Friendly insurance assistant.

You are a friendly insurance assistant.

STRICT RULES:
- Answer ONLY using the information provided below.
- Do NOT use outside knowledge.
- Do NOT guess or assume.
- If the answer is NOT available in the provided information, reply exactly:
  "Sorry, I don't know that. Is there any other insurance-related question you would like to talk about?"
- Keep the answer polite, clear, and well-polished.
- The answer must be within three lines.
- Do NOT mention documents, context, sources, or internal information.



CONTEXT:
{context}
"""
    ),
    ("placeholder", "{chat_history}"),
    ("human", "{question}")
])


In [19]:
rag_chain = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],
        "chat_history": lambda x: x.get("chat_history", [])
    }
    | prompt
    | llm
)



In [20]:

chat_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,           # session-aware memory
    input_messages_key="question",
    history_messages_key="chat_history",
)


d:\insurance-rag-chatbot\ragas_env\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [22]:
import uuid

def get_or_create_session_id(session_id: str | None):
    if session_id and session_id.strip():
        return session_id          
    return str(uuid.uuid4())       

In [23]:
incoming_session_id = None  


In [24]:
session_id = get_or_create_session_id(incoming_session_id)


In [25]:
def safe_chat_invoke(chat_chain, question, session_id):
    try:
        response = chat_chain.invoke(
            {"question": question},
            config={"configurable": {"session_id": session_id}}
        )
        return {"answer": response.content, "error": None}

    except Exception as e:
        error_msg = str(e).lower()

        if "rate limit" in error_msg:
            return {
                "answer": None,
                "error": "I'm temporarily busy due to high usage. Please try again shortly."
            }

        if "api key" in error_msg or "authentication" in error_msg:
            return {
                "answer": None,
                "error": "There is a configuration issue. Please try again later."
            }

        return {
            "answer": None,
            "error": "Something went wrong. Please try again."
        }


In [26]:
result = safe_chat_invoke(
    chat_chain,
    "What is health insurance?",
    session_id
)

print(result)


{'answer': "Health insurance is a type of insurance that covers medical expenses. It's a contract between an insurer and an individual or group, where the insurer agrees to provide specified health insurance cover at a particular premium.", 'error': None}


In [27]:
%pip install openai-whisper sounddevice scipy


^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import sounddevice as sd
from scipy.io.wavfile import write
import whisper

AUDIO_PATH = os.path.join(os.getcwd(), "input.wav")

def record_audio(duration=5, sample_rate=16000):
    print("Speak now...")
    audio = sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype="int16"
    )
    sd.wait()
    write(AUDIO_PATH, sample_rate, audio)
    print("Recording finished")

model = whisper.load_model("base")

def audio_to_text():
    result = model.transcribe(AUDIO_PATH)
    return result["text"].strip()

# Run end-to-end
record_audio()
text = audio_to_text()
print("Recognized text:", text)


In [ ]:
# 1. Record and transcribe
record_audio()
text = audio_to_text()
print("Recognized text:", text)

# 2. Send text to LLM (safe)
result = safe_chat_invoke(
    chat_chain,
    text,
    session_id
)

# 3. Handle response
if result["error"]:
    print("Error:", result["error"])
else:
    print("Answer:", result["answer"])

    

In [ ]:
# 1. Record and transcribe
record_audio()
text = audio_to_text()
print("Recognized text:", text)

# 2. Send text to LLM (safe)
result = safe_chat_invoke(
    chat_chain,
    text,
    session_id
)

# 3. Handle response
if result["error"]:
    print("Error:", result["error"])
else:
    print("Answer:", result["answer"])


In [29]:
import os
%pwd

'd:\\insurance-rag-chatbot\\notebooks'

  Using cached openai_whisper-20250625-py3-none-any.whl
  Using cached cffi-2.0.0-cp310-cp310-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached cffi-2.0.0-cp310-cp310-win_amd64.whl (182 kB)
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 645.7 kB/s eta 0:00:04
   ------- -------------------------------- 0.5/2.7 MB 645.7 kB/s eta 0:00:04
   ----------- ---------------------------- 0.8/2.7 MB 633.2 kB/s eta 0:00:04
   ----------- ---------------------------- 0.8/2.7 MB 633.2 kB/s eta 0:00:04
   ----------- ---------------------------- 0.8/2.7 MB 633.2 kB/s eta 0:00:04
   --------------- ------------------------ 1.0/2.7 MB 572.0 kB/s eta 0:00:03
   ----

In [30]:
os.chdir("../")
%pwd

'd:\\insurance-rag-chatbot'

In [32]:

from src.rag.Vectorstore import VectorStoreManager

# 1. Load the vector store (class method)
vector_stores = VectorStoreManager.load_vectorstore(
    r"D:\insurance-rag-chatbot\notebooks\faiss_index"
)

print(" Vector store loaded")

# 2. Run a similarity search
query = "What does health insurance cover?"
results = vector_stores.similarity_search(query, k=3)

print(f"\n Query: {query}\n")

for i, doc in enumerate(results, start=1):
    print(f"Result {i}:")
    print(doc.page_content[:300])  # first 300 chars
    print("Metadata:", doc.metadata)
    print("-" * 50)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10299.76it/s]


 Vector store loaded

 Query: What does health insurance cover?

Result 1:
Q. What kinds of Health Insurance plans are available?
Metadata: {}
--------------------------------------------------
Result 2:
2
2. Health Insurance
The term ‘Health Insurance’ relates to a type of insurance that essentially covers your medical expenses. A health insurance policy like other policies is a contract between an insurer and an individual / group in which the insurer agrees to provide specified health insurance c
Metadata: {}
--------------------------------------------------
Result 3:
or hospital expense benefits, inc luding assured benefits and long- term care, travel insurance and personal accident cover. Thus the term ‘Health Insurance’ relates to a type of insurance that essentially covers one’s medical expenses. 107 | Page Accordingly a health insurance policy is a contract 
Metadata: {}
--------------------------------------------------


In [38]:
evaluation_questions = [
    {
        "question": "What is health insurance?",
        "expected_keywords": [
            "health insurance",
            "medical expenses",
            "hospitalization"
        ]
    },
    {
        "question": "What documents are required for a claim?",
        "expected_keywords": [
            "claim",
            "documents",
            "hospital"
        ]
    },
    {
        "question": "What is the waiting period?",
        "expected_keywords": [
            "waiting period"
        ]
    },
    {
        "question": "What are the exclusions in health insurance?",
        "expected_keywords": [
            "exclusions",
            "not covered"
        ]
    },
    {
        "question": "How is premium calculated?",
        "expected_keywords": [
            "premium",
            "age",
            "sum insured"
        ]
    }
]

In [37]:
def precision_at_k(retrieved_docs, expected_keywords):
    relevant = 0

    for doc in retrieved_docs:
        text = doc.page_content.lower()

        if any(keyword.lower() in text for keyword in expected_keywords):
            relevant += 1

    return relevant / len(retrieved_docs)

In [39]:
def recall_at_k(retrieved_docs, expected_keywords):
    found = set()

    for doc in retrieved_docs:
        text = doc.page_content.lower()

        for keyword in expected_keywords:
            if keyword.lower() in text:
                found.add(keyword)

    return len(found) / len(expected_keywords)

In [40]:
precision_scores = []
recall_scores = []

for sample in evaluation_questions:

    docs = vectorstore.max_marginal_relevance_search(
        sample["question"],
        k=3,
        fetch_k=20
    )

    p = precision_at_k(
        docs,
        sample["expected_keywords"]
    )

    r = recall_at_k(
        docs,
        sample["expected_keywords"]
    )

    precision_scores.append(p)
    recall_scores.append(r)

    print("="*60)
    print("Question :", sample["question"])
    print(f"Precision@3 : {p:.2f}")
    print(f"Recall@3    : {r:.2f}")

Question : What is health insurance?
Precision@3 : 1.00
Recall@3    : 1.00
Question : What documents are required for a claim?
Precision@3 : 1.00
Recall@3    : 0.67
Question : What is the waiting period?
Precision@3 : 0.33
Recall@3    : 1.00
Question : What are the exclusions in health insurance?
Precision@3 : 0.67
Recall@3    : 0.50
Question : How is premium calculated?
Precision@3 : 1.00
Recall@3    : 0.67


In [41]:
average_precision = sum(precision_scores) / len(precision_scores)
average_recall = sum(recall_scores) / len(recall_scores)

print("="*60)
print(f"Average Precision@3 : {average_precision:.3f}")
print(f"Average Recall@3    : {average_recall:.3f}")

Average Precision@3 : 0.800
Average Recall@3    : 0.767
